# Qwen MVP Stage 3 outer LM gradient smoke
Select Tesla T4, enable Internet and authorize GITHUB_TOKEN. Save & Run All.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import tempfile
import uuid

remote = 'https://github.com/ArcheLabs/mini-cells.git'
branch = 'research/minicells-08-runtime'
run_id = 'qwen-mvp-e3-smoke-' + uuid.uuid4().hex[:12]
output = Path('/kaggle/working') / run_id
output.mkdir()
with tempfile.TemporaryDirectory() as tmp:
    repo = Path(tmp) / 'repo'
    subprocess.run(['git', 'clone', '--depth=1', '--branch', branch, remote, str(repo)],
                   check=True, timeout=300)
    source = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
    sys.path.insert(0, str(repo))
    from scripts.research.publish_experiment_results import (
        ExperimentSpec, load_github_token, prepare_artifacts, push_results,
    )
    load_github_token('GITHUB_TOKEN')
    log = output / 'raw.log'
    try:
        with log.open('w') as stream:
            subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers==5.3.0'],
                           stdout=stream, stderr=subprocess.STDOUT, check=True, timeout=600)
            subprocess.run([sys.executable, str(repo / 'research/titans/qwen-mvp/stage3.py'),
                            str(output / 'smoke.json')], cwd=repo, stdout=stream, stderr=subprocess.STDOUT,
                           check=True, timeout=1800)
        decision = json.loads((output / 'smoke.json').read_text())
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        decision = {'status': 'E3_GRADIENT_SMOKE_FAILED', 'error': str(exc)}
    result_branch = 'kaggle/' + run_id
    decision.update(format='minicells.qwen-mvp.e3-smoke.v1', source_commit=source, run_id=run_id)
    (output / 'decision.json').write_text(json.dumps(decision, indent=2))
    stage = repo / 'results/qwen-mvp-e3-smoke'
    stage.mkdir(parents=True)
    names = ('decision.json', 'raw.log')
    for name in names:
        shutil.copy2(output / name, stage / name)
    spec = ExperimentSpec('results/qwen-mvp-e3-smoke', 'artifacts/experiments/minicells08/' + run_id,
                          names, result_branch, decision['format'])
    dest = prepare_artifacts(repo, 'minicells08', spec, None)
    push_results(repo, dest, 'minicells08', result_branch, 'GITHUB_TOKEN', 'origin/main')
    receipt = {'status': 'REMOTE_SHA_VERIFIED', 'branch': result_branch,
               'commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()}
    (output / 'publication.json').write_text(json.dumps(receipt, indent=2))
    print(decision, receipt, flush=True)
    if decision['status'] != 'E3_GRADIENT_SMOKE_PASS':
        raise RuntimeError(f'Inspect {log}')
